# Capdyssey 실시간 분석 서버
이 노트북은 PC 화면에서 전송한 10초 PPG·IMU 데이터를 분석합니다.
런타임을 **GPU**로 설정하세요. 원래 RF/AF 모델을 재학습하지 않습니다.

무료 Colab은 외부 UI를 주로 사용하는 작업을 제한할 수 있습니다. 유료 플랜과 양수 컴퓨팅 잔액이 있는 대화형 실험 또는 별도 GPU 환경을 사용하세요. 상시 서버/세션 유지용 코드가 아닙니다.
실제 센서 정확도·IMU 임계값·Colab 연결은 현장 검증이 필요합니다.


In [ ]:
# 1. 전체 Capdyssey_realtime_v1.zip 업로드 및 압축 해제
from google.colab import files
from pathlib import Path
import zipfile, io
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('전체 패키지 ZIP 하나만 업로드하세요.')
data = next(iter(uploaded.values()))
destination = Path('/content/capdyssey_live_upload').resolve()
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data)) as z:
    if sum(x.file_size for x in z.infolist()) > 100_000_000:
        raise ValueError('예상보다 큰 ZIP입니다.')
    for info in z.infolist():
        target = (destination / info.filename).resolve()
        if not target.is_relative_to(destination):
            raise ValueError('잘못된 ZIP 경로')
    z.extractall(destination)
ROOT = destination / 'Capdyssey_realtime_v1'
assert (ROOT / 'model_bundle' / 'manifest.json').is_file(), '다른 ZIP을 업로드했습니다.'
print('패키지 준비 완료:', ROOT)


In [ ]:
# 2. 분석 서버 의존성 설치. CUDA용 torch는 Colab 기본 설치를 사용합니다.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-colab.txt')])
print('설치 후 런타임 재시작 안내가 나오면 재시작하고 3번 셀을 실행하세요.')


In [ ]:
# 3. 모델 검증·로딩. 재시작 후에도 업로드된 파일 경로를 사용합니다.
from pathlib import Path
import sys, os, importlib.metadata
ROOT = Path('/content/capdyssey_live_upload/Capdyssey_realtime_v1')
assert ROOT.is_dir(), '1번 업로드 셀을 먼저 실행하세요.'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
sys.path.insert(0, str(ROOT))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 필요합니다. 런타임 유형을 변경하세요.')
from realtime.analysis import AnalysisService, build_app
service = AnalysisService(ROOT / 'model_bundle')
print('모델 로딩 완료. GPU:', torch.cuda.get_device_name(0))
for package in ['numpy','scipy','scikit-learn','torch','gradio']:
    print(package, importlib.metadata.version(package))


In [ ]:
# 4. 인증된 실험용 연결 열기. 실행할 때마다 새 비밀번호를 생성합니다.
import secrets
password = secrets.token_urlsafe(18)
print('PC 프로그램의 분석 비밀번호 칸에 입력:', password)
print('아래 출력의 https://...gradio.live 주소를 PC 프로그램에 입력하세요.')
app = build_app(service)
app.queue(max_size=4).launch(share=True, auth=('capdyssey', password), show_error=False)


## 종료
PC에서 측정을 중지한 후 아래 셀을 실행하세요. 결과를 포함한 노트북을 공개할 때 공유 주소와 비밀번호 출력은 지우세요.


In [ ]:
app.close()
